# **Comportement de recherche de soins**
# Données issues des Enquêtes Démographiques et de Santé (EDS - DHS)

Ce rapport génère des visualisations pour les indicateurs relatifs à la recherche de soins pour enfants fébriles, à partir des données de l'Enquête Démographique et de Santé (EDS - DHS).

L'indicateur mesure, parmi les enfants de moins de 5 ans ayant présenté de la fièvre au cours des deux semaines précédant l'enquête, le pourcentage pour lesquels un avis médical ou un traitement a été sollicité.

La recherche de soins a été, dans le cadre du processus SNT, groupée en trois catégories:
- **soins publics**
- **soins privés**
- **aucun soin** relevant de la médecine moderne

---

* *Numérateur* : le nombre d'enfants vivants âgés de moins de 5 ans, ayant présenté de la fièvre à un moment quelconque au cours des deux semaines précédant l'entretien et pour lesquels des conseils ou un traitement ont été sollicités

* *Dénominateur* : le nombre d'enfants vivants âgés de moins de 5 ans ayant présenté de la fièvre à un moment quelconque au cours des deux semaines précédant l'enquête

---

Pour plus d'informations (en anglais):
- Ressources relatives aux comportements de recherche de soins
    - [Définition et calculs](https://dhsprogram.com/data/Guide-to-DHS-Statistics/Fever_and_Careseeking.htm?rhtocid=_13_3_0#Percentage_of_children4)
- [Les questionnaires utilisés dans les EDS/DHS](https://dhsprogram.com/publications/publication-dhsg4-dhs-questionnaires-and-manuals.cfm)

---

*Note* : Contrairement à la majorité des analyses dans le cadre du processus SNT, cette analyse est menée au niveau administratif **ADM1**, en raison de la disponibilité des données.

## 1. Configuration

In [ ]:
rm(list = ls())

options(scipen=999)

In [ ]:
# Global paths
Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")
Sys.setenv(GDAL_DATA = "/opt/conda/share/gdal")

# Paths
ROOT_PATH <- '~/workspace'
PIPELINE_PATH <- file.path(ROOT_PATH, 'pipelines', 'snt_dhs_indicators')
CONFIG_PATH <- file.path(ROOT_PATH, 'configuration')
CODE_PATH <- file.path(ROOT_PATH, 'code')
DATA_PATH <- file.path(ROOT_PATH, 'data')
DHS_DATA_PATH <- file.path(DATA_PATH, 'dhs', 'raw')
OUTPUT_DATA_PATH <- file.path(DATA_PATH, 'dhs', 'indicators', 'careseeking')
OUTPUT_PLOTS_PATH <- file.path(ROOT_PATH, 'pipelines', 'snt_dhs_indicators', 'reporting', 'outputs')

In [ ]:
# Load notebook-specific utilities
source(file.path(CODE_PATH, "snt_utils.r"))
source(file.path(CODE_PATH, "snt_report.r"))
source(file.path(CODE_PATH, "snt_palettes.r"))
source(file.path(PIPELINE_PATH, "utils", "snt_dhs_careseeking_report.r"))

# List required pcks
required_packages <- c("sf", "glue", "data.table", "ggplot2", "stringi", "jsonlite", "httr", "reticulate", "arrow", "IRdisplay")

# Execute function
install_and_load(required_packages)

In [ ]:
Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")
reticulate::py_config()$python
openhexa <- import("openhexa.sdk")

# Load SNT config
CONFIG_FILE_NAME <- "SNT_config.json"
config_json <- tryCatch({ fromJSON(file.path(CONFIG_PATH, CONFIG_FILE_NAME)) },
                        error = function(e) {
                          msg <- paste0("Error while loading configuration", conditionMessage(e))  
                          cat(msg)   
                          stop(msg) 
                        })

msg <- paste0("SNT configuration : ", file.path(CONFIG_PATH, CONFIG_FILE_NAME)) 
log_msg(msg)

# Set config variables
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE

In [ ]:
data_source <- 'DHS'
# indicator_public_care <- 'PUBLIC_CARE'
# indicator_private_care <- 'PRIVATE_CARE'
# indicator_no_care <- 'NO_CARE'

## 2. Chargement et pré-processing des données à visualiser

**Les données utilisées**

Toutes les données utilisées dans ce rapport sont agrégées au niveau administratif ADM1:

* Données spatiales : fond de carte (DHIS2)
* Données calculées par le pipeline:
   - valeurs estimées de recours aux soins pour les enfants fébriles, ainsi que sur la précision statistique de ces estimations (intervalles de confiance à 95%) : 
      - soins dans les services médicaux publics
      - soins dans les services médicaux privés
      - aucun soin, ou soins non-médicaux

In [ ]:
admin_level <- 'ADM1'
admin_id_col <- glue(admin_level, 'ID', .sep='_')
admin_name_col <- glue(admin_level, 'NAME', .sep='_')
admin_cols <- c(admin_id_col, admin_name_col)

In [ ]:
# Load spatial file from dataset

dhis2_dataset <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED

spatial_data_filename <- paste(COUNTRY_CODE, "shapes.geojson", sep = "_")
# spatial_data <- read_sf(file.path(DATA_PATH, 'dhis2', 'formatted', spatial_data_filename))
spatial_data <- get_latest_dataset_file_in_memory(dhis2_dataset, spatial_data_filename)
log_msg(glue("File {spatial_data_filename} successfully loaded from dataset version: {dhis2_dataset}"))

spatial_data <- st_as_sf(spatial_data)

# aggregate geometries by the admin columns
spatial_data <- aggregate_geometry(
  sf_data=spatial_data,
  admin_id_colname=admin_id_col,
  admin_name_colname=admin_name_col
)

# keep class
spatial_data <- st_as_sf(spatial_data)

if(COUNTRY_CODE == "COD"){
  spatial_data[[admin_name_col]] <- clean_admin_names(spatial_data[[admin_name_col]])
}

In [ ]:
filename_without_extension <- glue("{COUNTRY_CODE}_{data_source}_{admin_level}_PCT_CARESEEKING_SAMPLE_AVERAGE")
careseeking_table <- fread(file.path(OUTPUT_DATA_PATH, paste0(filename_without_extension, '.csv')))

# all columns which are not admin columns, are indicator columns
all_indicators <- setdiff(names(careseeking_table), admin_cols)

## 3. Création de graphiques

Pour chacun des trois indicateurs, le rapport crée deux visualisations:
   - Une carte choroplèthe indiquant l'estimation moyenne de la valeur de l'indicateur, sur base des données d'enquête
   - Un graphique de son intervalle de confiance, avec des barres d’erreur pour chaque ADM1

In [ ]:
# emtpy vectors to save locations of all files to render
careseeking_map_paths <- character(0)
careseeking_ci_paths <- character(0)

In [ ]:
# The maps

plot_data =  merge(spatial_data, careseeking_table, by = admin_cols, all = TRUE)

for (indicator_name in all_indicators) {

  # plot variables
  plot_label <- gsub("PCT ", "", gsub("_", " ", indicator_name))
  
  # build the map
  indicator_plot <- make_pct_choropleth_map(
    map_data = plot_data,
    target_colname = indicator_name,
    plot_title = glue("Soins pour enfants fébriles: {plot_label}"),
    plot_subtitle = COUNTRY_CODE,
    plot_caption = glue("Données: {data_source}"),
    legend_title = "%"
  )

  # save the map
  plot_path <- file.path(
      OUTPUT_PLOTS_PATH,
      glue::glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{toupper(indicator_name)}_plot.png")
      )
    
  suppressMessages(ggsave(
      plot = indicator_plot,
      filename = plot_path,
      width = 6,
      height = 5,
      dpi = 300
  ))

  # add map's location to vector of maps to render
  careseeking_map_paths <- c(careseeking_map_paths, plot_path)
}

In [ ]:
# The confidence interval plots
for (indicator_name in all_indicators) {

  # read in the data for that type of care
  ci_data <- data.table::fread(
    file.path(OUTPUT_DATA_PATH, glue::glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{indicator_name}.csv"))
  )

  # plot variables
  indicator_label <- gsub("_", " ", indicator_name)
  sample_avg_col <- glue::glue("{indicator_name}_SAMPLE_AVERAGE")
  lower_bound_col <- glue::glue("{indicator_name}_CI_LOWER_BOUND")
  upper_bound_col <- glue::glue("{indicator_name}_CI_UPPER_BOUND")
  
  # build plot
  ci_plot <- make_ci_plot(
    df_to_plot = ci_data,
    admin_colname = admin_name_col,
    point_estimation_colname = sample_avg_col,
    ci_lower_colname = lower_bound_col,
    ci_upper_colname = upper_bound_col,
    plot_title = glue::glue("{indicator_label} (Intervalles de confiance 95%)"),
    plot_subtitle = COUNTRY_CODE,
    plot_caption = glue::glue("Données : {data_source}"),
    x_title = admin_level,
    y_title = glue::glue("{indicator_label} (%)")
  )

  # save plot
  plot_path <- file.path(
    OUTPUT_PLOTS_PATH,
    glue::glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{toupper(indicator_name)}_CI_plot.png")
    )

  suppressMessages(ggsave(
    plot = ci_plot,
    filename = plot_path,
    width = 6,
    height = 5,
    dpi = 300
  ))

  # add plot's location to vector of ci plots to render
  careseeking_ci_paths <- c(careseeking_ci_paths, plot_path)
}

In [ ]:
# display all maps and confidence interval plots, avoiding lapply's NULL returns
invisible(lapply(careseeking_map_paths, function(p) display_png(file = p)))
invisible(lapply(careseeking_ci_paths, function(p) display_png(file = p)))